# 03a0 - Cache Eagle2-2B For Offline RTX6000 Encode

Run this notebook on Kaggle with internet enabled. It downloads the Eagle2-2B Hugging Face snapshot and the Python wheelhouse needed by Notebook 03a offline.

Main output:
- `eagle2_offline_cache/hf_models/nvidia_Eagle2-2B`
- `eagle2_offline_cache/wheelhouse`
- `eagle2_offline_cache/cache_report.json`

Use the output of this notebook as an input to `03a_encode_vlm_features_offline_rtx6000.ipynb`.

In [1]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

CACHE_ROOT = Path("/kaggle/working/eagle2_offline_cache")
MODEL_DIR = CACHE_ROOT / "hf_models" / "nvidia_Eagle2-2B"
WHEELHOUSE = CACHE_ROOT / "wheelhouse"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
WHEELHOUSE.mkdir(parents=True, exist_ok=True)

print("CACHE_ROOT:", CACHE_ROOT)

CACHE_ROOT: /kaggle/working/eagle2_offline_cache


In [2]:
# Keep versions aligned with Eagle2 remote code.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-cache-dir",
    "huggingface_hub>=0.30.0,<1.0",
])

from huggingface_hub import snapshot_download

def load_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if token:
        return token.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            return token.strip()
    except Exception as exc:
        print(f"HF token lookup skipped/failed: {type(exc).__name__}: {exc}")
    return None

HF_TOKEN = load_hf_token()
print("HF token available:", bool(HF_TOKEN))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 74.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.


HF token lookup skipped/failed: ConnectionError: Connection error trying to communicate with service.
HF token available: False


In [3]:
start = time.time()
model_path = snapshot_download(
    repo_id="nvidia/Eagle2-2B",
    local_dir=str(MODEL_DIR),
    local_dir_use_symlinks=False,
    token=HF_TOKEN,
    ignore_patterns=[".git/*", "*.md"],
)
print("Downloaded model snapshot to:", model_path)
print("elapsed_sec:", round(time.time() - start, 1))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

chat_template.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

configuration_eagle2_5_vl.py: 0.00B [00:00, ?B/s]

...

processor_config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Downloaded model snapshot to: /kaggle/working/eagle2_offline_cache/hf_models/nvidia_Eagle2-2B
elapsed_sec: 18.4


In [4]:
# Build an offline wheelhouse for the RTX6000 notebook.
# Do not download torch; Kaggle GPU images already provide a CUDA PyTorch build.
requirements = [
    "transformers==4.51.0",
    "huggingface_hub>=0.30.0,<1.0",
    "accelerate>=1.7.0",
    "decord>=0.6.0",
    "av",
    "opencv-python-headless",
]
cmd = [sys.executable, "-m", "pip", "download", "-q", "--dest", str(WHEELHOUSE), *requirements]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)
print("wheelhouse files:", len(list(WHEELHOUSE.iterdir())))

Running: /usr/bin/python3 -m pip download -q --dest /kaggle/working/eagle2_offline_cache/wheelhouse transformers==4.51.0 huggingface_hub>=0.30.0,<1.0 accelerate>=1.7.0 decord>=0.6.0 av opencv-python-headless
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 9.9 MB/s eta 0:

In [5]:
def dir_size_bytes(path):
    total = 0
    for p in Path(path).rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total

report = {
    "cache_version": "eagle2_2b_offline_cache_v1",
    "model_id": "nvidia/Eagle2-2B",
    "cache_root": str(CACHE_ROOT),
    "model_dir": str(MODEL_DIR),
    "wheelhouse": str(WHEELHOUSE),
    "model_size_gb": round(dir_size_bytes(MODEL_DIR) / 1024**3, 3),
    "wheelhouse_size_gb": round(dir_size_bytes(WHEELHOUSE) / 1024**3, 3),
    "num_model_files": sum(1 for p in MODEL_DIR.rglob("*") if p.is_file()),
    "num_wheel_files": sum(1 for p in WHEELHOUSE.iterdir() if p.is_file()),
}
(CACHE_ROOT / "cache_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

{
  "cache_version": "eagle2_2b_offline_cache_v1",
  "model_id": "nvidia/Eagle2-2B",
  "cache_root": "/kaggle/working/eagle2_offline_cache",
  "model_dir": "/kaggle/working/eagle2_offline_cache/hf_models/nvidia_Eagle2-2B",
  "wheelhouse": "/kaggle/working/eagle2_offline_cache/wheelhouse",
  "model_size_gb": 4.129,
  "wheelhouse_size_gb": 2.689,
  "num_model_files": 37,
  "num_wheel_files": 49
}
